# Appendix Tables

Generates per-model LaTeX tables for:
1. **Cumulative regret@10** — `high_scale` and `high_neg_scale`, one table per model per scale.
2. **Exploration count @3 and @10** — both scales, one table per model per scale.
3. **Full exploration & consecutive exploration** — proportion of replicates that try all 3 arms, and (conditional) proportion where the 2nd and 3rd new arm trials occur on consecutive turns.

In [1]:
import re
import os
import numpy as np
import pandas as pd

In [2]:
os.chdir("../..")

## Lookup Maps and Style Constants

In [3]:
rewards_per_scale = {
    'high_scale':     [75, 50, 25],
    'low_scale':      [0.75, 0.5, 0.25],
    'high_neg_scale': [-25, -50, -75],
    'low_neg_scale':  [-0.25, -0.5, -0.75],
}

nomenclature_map = {
    'alphanumeric':     'Alphanumeric',
    'ordinal_helpful':  'Ordinal - Helpful',
    'ordinal_mislead':  'Ordinal - Misleading',
    'world_helpful':    'World - Helpful',
    'world_mislead':    'World - Misleading',
    'sent_new_helpful': 'Sentiment - Helpful',
    'sent_new_mislead': 'Sentiment - Misleading',
}

## Load Data

In [4]:
results_df = pd.read_csv('experiments/merged_sem_var_results.csv')
print(f"Loaded {len(results_df)} rows")
results_df[['domain', 'model', 'history', 'variance']].value_counts().reset_index(name='count')

Loaded 1769 rows


,domain,model,history,variance,count
0,Farm,Qwen3-32B,Summarized,No,64
1,Clothing Recommendation,Olmo-3.1-32B-Instruct,Summarized,No,56
2,Clothing Recommendation,Qwen3-32B,Summarized,No,56
3,Farm,Olmo-3.1-32B-Instruct,Summarized,No,56
4,Clothing Recommendation,gemini-3.1-flash-lite-preview,Summarized,No,56
...,...,...,...,...,...
59,Classical,TS,Symbolic,No,2
60,Classical,UCB1,Symbolic,High,2
61,Classical,UCB1,Symbolic,Low,2
62,Classical,UCB1,Symbolic,No,2


## Configuration

In [5]:
HIST   = 'Summarized'

MODELS_TO_INCLUDE = [
    ('Qwen3-32B',                    'Qwen3-32B'),
    ('Olmo-3.1-32B-Instruct',        'OLMo-3.1 32B'),
    ('gemini-3.1-flash-lite-preview', 'Gemini 3.1 Flash Lite'),
]

# Nomenclature rows: (display_label, helpful_key, mislead_key)
# None for mislead_key = no misleading counterpart
NOM_ROWS = [
    ('Ordinal',      'ordinal_helpful',  'ordinal_mislead'),
    ('World',        'world_helpful',    'world_mislead'),
    ('Sentiment',    'sent_new_helpful', 'sent_new_mislead'),
    ('Alphanumeric', 'alphanumeric',     None),
]

DOMAIN_ORDER   = ['Bandit', 'Farm', 'Clothing Recommendation']
DOMAIN_DISPLAY = {'Bandit': 'Bandit', 'Farm': 'Farm', 'Clothing Recommendation': 'Clothing'}
VARIANCE_ORDER = ['No', 'Low', 'High']

# Scales shown as rows; labels use H/L (magnitude) and +/- (sign)
SCALES = ['high_scale', 'low_scale', 'low_neg_scale', 'high_neg_scale']
SCALE_LABEL = {
    'high_scale':     'H+',
    'low_scale':      'L+',
    'low_neg_scale':  'L-',
    'high_neg_scale': 'H-',
}

# Turn indices: index 2 = 3rd action (@3); index 9 = 10th/last action (@10)
EXPLOR_TURNS = [2, 9]

## Part 1: Cumulative Regret Tables

One table per model per scale. Rows = (variance, nomenclature). Columns = domain × {Helpful, Misleading}.

In [6]:
turn_numbers = sorted(
    int(col.split('_')[-2])
    for col in results_df.columns
    if col.startswith('cum_regret_') and col.endswith('_mean')
)
last_turn = max(turn_numbers)


def fmt(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '--'
    return f'{val:.2f}'


def build_regret_table(df, present_domains):
    """Per-model, per-variance regret table. Rows = (scale, nomenclature). Columns = domain × {H, M}."""
    col_spec = 'lr' + ''.join(['|rr' for _ in present_domains])

    # Bold targets per (domain, scale): min H, max M  (exclude classical baselines)
    llm_df = df[df['Nomenclature'] != '']
    bold_H, bold_M = {}, {}
    for domain in present_domains:
        dcol = DOMAIN_DISPLAY[domain]
        for scale_lbl in llm_df['Scale'].unique():
            mdf    = llm_df[llm_df['Scale'] == scale_lbl]
            h_vals = mdf[f'{dcol} H'].dropna()
            m_vals = mdf[f'{dcol} M'].dropna()
            if not h_vals.empty:
                bold_H[(dcol, scale_lbl)] = h_vals.min()
            if not m_vals.empty:
                bold_M[(dcol, scale_lbl)] = m_vals.max()

    def fmt_bold(val, target):
        s = fmt(val)
        if s == '--' or target is None:
            return s
        if abs(float(val) - target) < 1e-9:
            return r'\textbf{' + s + r'}'
        return s

    lines = []
    lines.append(r'\resizebox{\columnwidth}{!}{%')
    lines.append(r'\begin{tabular}{' + col_spec + r'}')
    lines.append(r'\toprule')

    # Header row 1: domain group spans
    header1 = ['', '']
    for domain in present_domains:
        header1.append(r'\multicolumn{2}{c}{' + DOMAIN_DISPLAY[domain] + r'}')
    lines.append(' & '.join(header1) + r' \\')

    # Cmidrules
    cmidrules = []
    for i in range(len(present_domains)):
        c1 = 3 + i * 2
        c2 = c1 + 1
        cmidrules.append(r'\cmidrule(lr){' + f'{c1}-{c2}' + r'}')
    lines.append(' '.join(cmidrules))

    # Header row 2
    header2 = ['Scale', 'Nomenclature']
    for _ in present_domains:
        header2 += ['Helpful', 'Misleading']
    lines.append(' & '.join(header2) + r' \\')
    lines.append(r'\midrule')

    prev_scale = None
    for _, row in df.iterrows():
        is_new_scale = (row['Scale'] != prev_scale)
        is_classical = (row['Nomenclature'] == '')

        if prev_scale is not None and is_new_scale:
            lines.append(r'\midrule')

        scale_cell = row['Scale'] if is_new_scale else ''
        prev_scale = row['Scale']

        cells = [scale_cell, row['Nomenclature']]
        for domain in present_domains:
            dcol  = DOMAIN_DISPLAY[domain]
            h_val = row.get(f'{dcol} H')
            m_val = row.get(f'{dcol} M')
            if is_classical:
                cells.append(fmt(h_val))
                cells.append(fmt(m_val))
            else:
                key = (dcol, row['Scale'])
                cells.append(fmt_bold(h_val, bold_H.get(key)))
                cells.append(fmt_bold(m_val, bold_M.get(key)))
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}%')
    lines.append(r'}')
    return '\n'.join(lines)


SCALE_CAPTION = (
    r'Scale labels: H\,=\,high reward magnitude; L\,=\,low; '
    r'+\,=\,positive rewards; $-$\,=\,negative '
    r'(e.g.\ H+ is the high positive scale).'
)


def wrap_regret_table(tabular_str, model_label, variance):
    model_safe = model_label.replace(' ', '_').replace('.', '')
    return '\n'.join([
        r'\begin{table}[H]',
        r'\centering',
        r'\small',
        tabular_str,
        r'\caption{Normalized cumulative regret at the final turn for ' + model_label
        + r', variance condition: ' + variance
        + r', by reward scale and nomenclature type. ' + SCALE_CAPTION + r'}',
        r'\label{tab:regret_' + model_safe + '_var' + variance + r'}',
        r'\end{table}',
    ])


os.makedirs('figures/tables', exist_ok=True)

for model_id, model_label in MODELS_TO_INCLUDE:
    for variance in VARIANCE_ORDER:
        mask = (
            (results_df['history']  == HIST) &
            (results_df['model']    == model_id) &
            (results_df['variance'] == variance)
        )
        df_var = results_df[mask].copy()
        present_domains = [d for d in DOMAIN_ORDER if d in df_var['domain'].unique()]

        rows = []
        for scale in SCALES:
            scale_lbl = SCALE_LABEL[scale]
            norm_const = (
                max(rewards_per_scale[scale]) - min(rewards_per_scale[scale])
            ) * len(turn_numbers)
            df_scale = df_var[df_var['scale'] == scale]

            for nom_label, helpful_key, mislead_key in NOM_ROWS:
                row = {'Scale': scale_lbl, 'Nomenclature': nom_label}
                for domain in present_domains:
                    dcol       = DOMAIN_DISPLAY[domain]
                    dom_subset = df_scale[df_scale['domain'] == domain]
                    for col_suffix, key in [('H', helpful_key), ('M', mislead_key)]:
                        if key is None:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                            continue
                        nom_df = dom_subset[dom_subset['nomenclature'].str.contains(key, na=False)]
                        if not nom_df.empty:
                            val = nom_df[f'cum_regret_{last_turn}_mean'].values[0]
                            row[f'{dcol} {col_suffix}'] = val / norm_const
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                rows.append(row)

        table_df = pd.DataFrame(rows)
        tabular  = build_regret_table(table_df, present_domains)
        latex    = wrap_regret_table(tabular, model_label, variance)

        path = f'figures/tables/regret_table_{model_id}_var{variance}.tex'
        with open(path, 'w') as f:
            f.write(latex)
        print(f'Saved: {path}')

print('\nDone.')

Saved: figures/tables/regret_table_Qwen3-32B_varNo.tex
Saved: figures/tables/regret_table_Qwen3-32B_varLow.tex
Saved: figures/tables/regret_table_Qwen3-32B_varHigh.tex
Saved: figures/tables/regret_table_Olmo-3.1-32B-Instruct_varNo.tex
Saved: figures/tables/regret_table_Olmo-3.1-32B-Instruct_varLow.tex
Saved: figures/tables/regret_table_Olmo-3.1-32B-Instruct_varHigh.tex
Saved: figures/tables/regret_table_gemini-3.1-flash-lite-preview_varNo.tex
Saved: figures/tables/regret_table_gemini-3.1-flash-lite-preview_varLow.tex
Saved: figures/tables/regret_table_gemini-3.1-flash-lite-preview_varHigh.tex

Done.


## Part 2: Exploration Count Tables

One table per model per scale. Rows = (variance, nomenclature) + UCB1 baseline.  
Columns per domain: H@3 | H@10 | M@3 | M@10.

In [7]:
def fmt_count(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '--'
    return f'{val:.1f}'


def build_exploration_table(df, present_domains):
    """Per-model, per-variance exploration table. Rows = (scale, nomenclature). Columns = domain × {H@3,H@10,M@3,M@10}."""
    col_spec = 'lr' + ''.join(['|rrrr' for _ in present_domains])

    # Bold targets per (domain, scale): max H@10, max M@10 (exclude classical baselines)
    llm_df = df[~df['Nomenclature'].isin(['UCB1', ''])]
    bold_H10, bold_M10 = {}, {}
    for domain in present_domains:
        dcol = DOMAIN_DISPLAY[domain]
        for scale_lbl in llm_df['Scale'].unique():
            mdf     = llm_df[llm_df['Scale'] == scale_lbl]
            h10vals = mdf[f'{dcol} H@10'].dropna()
            m10vals = mdf[f'{dcol} M@10'].dropna()
            if not h10vals.empty:
                bold_H10[(dcol, scale_lbl)] = h10vals.max()
            if not m10vals.empty:
                bold_M10[(dcol, scale_lbl)] = m10vals.max()

    def fmt_bold_count(val, target):
        s = fmt_count(val)
        if s == '--' or target is None:
            return s
        if abs(float(val) - target) < 1e-9:
            return r'\textbf{' + s + r'}'
        return s

    lines = []
    lines.append(r'\resizebox{\columnwidth}{!}{%')
    lines.append(r'\begin{tabular}{' + col_spec + r'}')
    lines.append(r'\toprule')

    # Header row 1: domain group spans (4 cols each)
    header1 = ['', '']
    for domain in present_domains:
        header1.append(r'\multicolumn{4}{c}{' + DOMAIN_DISPLAY[domain] + r'}')
    lines.append(' & '.join(header1) + r' \\')

    # Cmidrules for domain groups
    dom_cmidrules = []
    for i in range(len(present_domains)):
        c1 = 3 + i * 4
        c2 = c1 + 3
        dom_cmidrules.append(r'\cmidrule(lr){' + f'{c1}-{c2}' + r'}')
    lines.append(' '.join(dom_cmidrules))

    # Header row 2: Helpful (span 2) | Misleading (span 2) per domain
    header2 = ['', '']
    hm_cmidrules = []
    for i, domain in enumerate(present_domains):
        c_h1 = 3 + i * 4
        c_h2 = c_h1 + 1
        c_m1 = c_h2 + 1
        c_m2 = c_m1 + 1
        header2.append(r'\multicolumn{2}{c}{Helpful}')
        header2.append(r'\multicolumn{2}{c}{Misleading}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_h1}-{c_h2}' + r'}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_m1}-{c_m2}' + r'}')
    lines.append(' & '.join(header2) + r' \\')
    lines.append(' '.join(hm_cmidrules))

    # Header row 3
    header3 = ['Scale', 'Nomenclature']
    for _ in present_domains:
        header3 += [r'@3', r'@10', r'@3', r'@10']
    lines.append(' & '.join(header3) + r' \\')
    lines.append(r'\midrule')

    prev_scale = None
    for _, row in df.iterrows():
        is_new_scale = (row['Scale'] != prev_scale)
        is_classical = row['Nomenclature'] in ['UCB1', '']

        if prev_scale is not None and is_new_scale:
            lines.append(r'\midrule')

        scale_cell = row['Scale'] if is_new_scale else ''
        prev_scale = row['Scale']

        cells = [scale_cell, row['Nomenclature']]
        for domain in present_domains:
            dcol = DOMAIN_DISPLAY[domain]
            if is_classical:
                cells.append(fmt_count(row.get(f'{dcol} H@3')))
                cells.append(fmt_count(row.get(f'{dcol} H@10')))
                cells.append(fmt_count(row.get(f'{dcol} M@3')))
                cells.append(fmt_count(row.get(f'{dcol} M@10')))
            else:
                key = (dcol, row['Scale'])
                cells.append(fmt_count(row.get(f'{dcol} H@3')))
                cells.append(fmt_bold_count(row.get(f'{dcol} H@10'), bold_H10.get(key)))
                cells.append(fmt_count(row.get(f'{dcol} M@3')))
                cells.append(fmt_bold_count(row.get(f'{dcol} M@10'), bold_M10.get(key)))
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}%')
    lines.append(r'}')
    return '\n'.join(lines)


def wrap_exploration_table(tabular_str, model_label, variance):
    model_safe = model_label.replace(' ', '_').replace('.', '')
    return '\n'.join([
        r'\begin{table}[H]',
        r'\centering',
        r'\small',
        tabular_str,
        r'\caption{Exploration count (unique arms tried) at turns 3 and 10 for ' + model_label
        + r', variance condition: ' + variance
        + r', by reward scale and nomenclature type. ' + SCALE_CAPTION + r'}',
        r'\label{tab:exploration_' + model_safe + '_var' + variance + r'}',
        r'\end{table}',
    ])


baseline_df = pd.read_csv('experiments/merged_sem_var_results.csv')

for model_id, model_label in MODELS_TO_INCLUDE:
    for variance in VARIANCE_ORDER:
        t3, t10 = EXPLOR_TURNS

        mask = (
            (results_df['history']  == HIST) &
            (results_df['model']    == model_id) &
            (results_df['variance'] == variance)
        )
        df_var = results_df[mask].copy()
        present_domains = [d for d in DOMAIN_ORDER if d in df_var['domain'].unique()]

        rows = []
        for scale in SCALES:
            scale_lbl = SCALE_LABEL[scale]
            df_scale  = df_var[df_var['scale'] == scale]

            for nom_label, helpful_key, mislead_key in NOM_ROWS:
                row = {'Scale': scale_lbl, 'Nomenclature': nom_label}
                for domain in present_domains:
                    dcol       = DOMAIN_DISPLAY[domain]
                    dom_subset = df_scale[df_scale['domain'] == domain]
                    for col_suffix, turn_idx, key in [
                        ('H@3',  t3,  helpful_key),
                        ('H@10', t10, helpful_key),
                        ('M@3',  t3,  mislead_key),
                        ('M@10', t10, mislead_key),
                    ]:
                        if key is None:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                            continue
                        nom_df = dom_subset[dom_subset['nomenclature'].str.contains(key, na=False)]
                        if not nom_df.empty:
                            row[f'{dcol} {col_suffix}'] = nom_df[f'exploration_count_{turn_idx}_mean'].values[0]
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                rows.append(row)

        # Classical baselines (UCB1) — domain='Classical', value replicated across domains
        for bl_model, bl_label in [('UCB1', 'UCB1')]:
            for scale in SCALES:
                scale_lbl = SCALE_LABEL[scale]
                bl_row_df = baseline_df[
                    (baseline_df['model']        == bl_model) &
                    (baseline_df['variance']     == variance) &
                    (baseline_df['scale']        == scale) &
                    (baseline_df['nomenclature'] == 'baseline')
                ]
                row = {'Scale': scale_lbl, 'Nomenclature': bl_label}
                for domain in present_domains:
                    dcol = DOMAIN_DISPLAY[domain]
                    for col_suffix, turn_idx in [('H@3', t3), ('H@10', t10)]:
                        if not bl_row_df.empty:
                            row[f'{dcol} {col_suffix}'] = bl_row_df[f'exploration_count_{turn_idx}_mean'].values[0]
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                    row[f'{dcol} M@3']  = float('nan')
                    row[f'{dcol} M@10'] = float('nan')
                rows.append(row)

        table_df = pd.DataFrame(rows)
        tabular  = build_exploration_table(table_df, present_domains)
        latex    = wrap_exploration_table(tabular, model_label, variance)

        path = f'figures/tables/exploration_count_table_{model_id}_var{variance}.tex'
        with open(path, 'w') as f:
            f.write(latex)
        print(f'Saved: {path}')

print('\nDone.')

Saved: figures/tables/exploration_count_table_Qwen3-32B_varNo.tex
Saved: figures/tables/exploration_count_table_Qwen3-32B_varLow.tex
Saved: figures/tables/exploration_count_table_Qwen3-32B_varHigh.tex
Saved: figures/tables/exploration_count_table_Olmo-3.1-32B-Instruct_varNo.tex
Saved: figures/tables/exploration_count_table_Olmo-3.1-32B-Instruct_varLow.tex
Saved: figures/tables/exploration_count_table_Olmo-3.1-32B-Instruct_varHigh.tex
Saved: figures/tables/exploration_count_table_gemini-3.1-flash-lite-preview_varNo.tex


Saved: figures/tables/exploration_count_table_gemini-3.1-flash-lite-preview_varLow.tex
Saved: figures/tables/exploration_count_table_gemini-3.1-flash-lite-preview_varHigh.tex

Done.


## Part 3: Full Exploration Tables

One table per model per variance. Rows = (scale, nomenclature). Columns = domain × {Helpful, Misleading} × {Full%, Consec%}.

- **Full%**: proportion of replicates that try all 3 arms by turn 10
- **Consec%**: among fully-exploring replicates, proportion where the 2nd and 3rd new arm trials happen on consecutive turns (t₂ − t₁ = 1)

In [8]:
import json

MODELS_TO_INCLUDE_RAW = [m for m, _ in MODELS_TO_INCLUDE]

rep_df = pd.read_csv('experiments/merged_replicate_results.csv')
rep_df = rep_df[
    (rep_df['history'] == HIST) &
    (rep_df['model'].isin(MODELS_TO_INCLUDE_RAW))
].copy()

print(f"Loaded {len(rep_df)} replicates")
rep_df[['domain', 'model', 'variance']].value_counts().reset_index(name='count')

Loaded 8887 replicates


,domain,model,variance,count
0,Farm,Qwen3-32B,No,640
1,Clothing Recommendation,Qwen3-32B,No,560
2,Clothing Recommendation,Olmo-3.1-32B-Instruct,No,560
3,Farm,Olmo-3.1-32B-Instruct,No,559
4,Clothing Recommendation,gemini-3.1-flash-lite-preview,No,544
5,Farm,gemini-3.1-flash-lite-preview,No,525
6,Bandit,Qwen3-32B,No,480
7,Bandit,Olmo-3.1-32B-Instruct,No,400
8,Bandit,gemini-3.1-flash-lite-preview,No,388
9,Farm,Qwen3-32B,Low,360


In [9]:
def compute_exploration_stats(jsonl_path):
    """Return (fully_explores, consecutive) for a single replicate JSONL file.

    fully_explores: tried all 3 arms by the final turn
    consecutive:    (only meaningful when fully_explores=True)
                    the turn reaching 3 unique arms = the turn reaching 2 unique arms + 1
    """
    with open(jsonl_path) as f:
        records = [json.loads(line) for line in f if line.strip()]

    seen = set()
    counts = []
    for r in records:
        seen.add(r['action_taken_id'])
        counts.append(len(seen))

    if not counts:
        return False, False

    fully = counts[-1] >= 3
    consec = False
    if fully:
        t1 = next(i for i, c in enumerate(counts) if c >= 2)
        t2 = next(i for i, c in enumerate(counts) if c >= 3)
        consec = (t2 - t1 == 1)
    return fully, consec


stats = rep_df['jsonl_file'].apply(lambda p: pd.Series(
    compute_exploration_stats(p),
    index=['fully_explores', 'consecutive']
))
rep_df = rep_df.join(stats)

print(f"fully_explores: {rep_df['fully_explores'].mean():.1%} overall")
rep_df[['fully_explores', 'consecutive']].describe()

fully_explores: 76.3% overall


,fully_explores,consecutive
count,8887,8887
unique,2,2
top,True,True
freq,6779,5955


In [10]:
GRP_KEYS = ['model', 'domain', 'nomenclature', 'scale', 'variance']

prop_full = (rep_df
    .groupby(GRP_KEYS)['fully_explores']
    .mean()
    .rename('prop_full')
    .reset_index())

prop_consec = (rep_df[rep_df['fully_explores']]
    .groupby(GRP_KEYS)['consecutive']
    .mean()
    .rename('prop_consec')
    .reset_index())

expl_stats = prop_full.merge(prop_consec, on=GRP_KEYS, how='left')

print(f"{len(expl_stats)} rows")
expl_stats.head()

668 rows


,model,domain,nomenclature,scale,variance,prop_full,prop_consec
0,Olmo-3.1-32B-Instruct,Bandit,alphanumeric,high_neg_scale,High,1.0,0.8
1,Olmo-3.1-32B-Instruct,Bandit,alphanumeric,high_neg_scale,Low,1.0,0.7
2,Olmo-3.1-32B-Instruct,Bandit,alphanumeric,high_neg_scale,No,1.0,0.9
3,Olmo-3.1-32B-Instruct,Bandit,alphanumeric,high_scale,High,0.6,0.5
4,Olmo-3.1-32B-Instruct,Bandit,alphanumeric,high_scale,Low,0.2,1.0


In [11]:
def fmt_pct(val):
    """Format a proportion as a percentage string (e.g. 0.75 → '75.0')."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '--'
    return f'{val * 100:.1f}'


def build_full_exploration_table(df, present_domains):
    """Per-model, per-variance full-exploration table.
    Rows = (scale, nomenclature). Columns = domain × {H,M} × {Full%, Consec%}.
    """
    col_spec = 'lr' + ''.join(['|rrrr' for _ in present_domains])

    # Bold target per (domain, scale): max Full% across LLM models (Helpful and Misleading)
    llm_df = df[df['Nomenclature'] != '']
    bold_H_full, bold_M_full = {}, {}
    for domain in present_domains:
        dcol = DOMAIN_DISPLAY[domain]
        for scale_lbl in llm_df['Scale'].unique():
            mdf = llm_df[llm_df['Scale'] == scale_lbl]
            h_vals = mdf[f'{dcol} H Full'].dropna()
            m_vals = mdf[f'{dcol} M Full'].dropna()
            if not h_vals.empty:
                bold_H_full[(dcol, scale_lbl)] = h_vals.max()
            if not m_vals.empty:
                bold_M_full[(dcol, scale_lbl)] = m_vals.max()

    def fmt_bold_pct(val, target):
        s = fmt_pct(val)
        if s == '--' or target is None:
            return s
        if abs(val - target) < 1e-9:
            return r'\textbf{' + s + r'}'
        return s

    lines = []
    lines.append(r'\resizebox{\columnwidth}{!}{%')
    lines.append(r'\begin{tabular}{' + col_spec + r'}')
    lines.append(r'\toprule')

    # Header row 1: domain group spans (4 cols each)
    header1 = ['', '']
    for domain in present_domains:
        header1.append(r'\multicolumn{4}{c}{' + DOMAIN_DISPLAY[domain] + r'}')
    lines.append(' & '.join(header1) + r' \\')

    dom_cmidrules = []
    for i in range(len(present_domains)):
        c1 = 3 + i * 4
        c2 = c1 + 3
        dom_cmidrules.append(r'\cmidrule(lr){' + f'{c1}-{c2}' + r'}')
    lines.append(' '.join(dom_cmidrules))

    # Header row 2: Helpful (span 2) | Misleading (span 2) per domain
    header2 = ['', '']
    hm_cmidrules = []
    for i in range(len(present_domains)):
        c_h1 = 3 + i * 4
        c_h2 = c_h1 + 1
        c_m1 = c_h2 + 1
        c_m2 = c_m1 + 1
        header2.append(r'\multicolumn{2}{c}{Helpful}')
        header2.append(r'\multicolumn{2}{c}{Misleading}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_h1}-{c_h2}' + r'}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_m1}-{c_m2}' + r'}')
    lines.append(' & '.join(header2) + r' \\')
    lines.append(' '.join(hm_cmidrules))

    # Header row 3
    header3 = ['Scale', 'Nomenclature']
    for _ in present_domains:
        header3 += [r'Full\%', r'Consec\%', r'Full\%', r'Consec\%']
    lines.append(' & '.join(header3) + r' \\')
    lines.append(r'\midrule')

    prev_scale = None
    for _, row in df.iterrows():
        is_new_scale = (row['Scale'] != prev_scale)

        if prev_scale is not None and is_new_scale:
            lines.append(r'\midrule')

        scale_cell = row['Scale'] if is_new_scale else ''
        prev_scale = row['Scale']

        cells = [scale_cell, row['Nomenclature']]
        for domain in present_domains:
            dcol = DOMAIN_DISPLAY[domain]
            key  = (dcol, row['Scale'])
            cells.append(fmt_bold_pct(row.get(f'{dcol} H Full'),  bold_H_full.get(key)))
            cells.append(fmt_pct(row.get(f'{dcol} H Consec')))
            cells.append(fmt_bold_pct(row.get(f'{dcol} M Full'),  bold_M_full.get(key)))
            cells.append(fmt_pct(row.get(f'{dcol} M Consec')))
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}%')
    lines.append(r'}')
    return '\n'.join(lines)


def wrap_full_exploration_table(tabular_str, model_label, variance):
    model_safe = model_label.replace(' ', '_').replace('.', '')
    return '\n'.join([
        r'\begin{table}[H]',
        r'\centering',
        r'\small',
        tabular_str,
        r'\caption{Full exploration rate and consecutive exploration rate for ' + model_label
        + r', variance condition: ' + variance
        + r'. Full\% = proportion of replicates that tried all 3 arms; '
        + r'Consec\% = among fully-exploring replicates, proportion where the 2nd and 3rd new arm trials occurred on consecutive turns. '
        + SCALE_CAPTION + r'}',
        r'\label{tab:full_exploration_' + model_safe + '_var' + variance + r'}',
        r'\end{table}',
    ])


for model_id, model_label in MODELS_TO_INCLUDE:
    for variance in VARIANCE_ORDER:
        sub = expl_stats[
            (expl_stats['model']    == model_id) &
            (expl_stats['variance'] == variance)
        ]
        present_domains = [d for d in DOMAIN_ORDER if d in sub['domain'].unique()]

        rows = []
        for scale in SCALES:
            scale_lbl = SCALE_LABEL[scale]
            df_scale  = sub[sub['scale'] == scale]

            for nom_label, helpful_key, mislead_key in NOM_ROWS:
                row = {'Scale': scale_lbl, 'Nomenclature': nom_label}
                for domain in present_domains:
                    dcol       = DOMAIN_DISPLAY[domain]
                    dom_subset = df_scale[df_scale['domain'] == domain]
                    for col_suffix, key in [('H', helpful_key), ('M', mislead_key)]:
                        if key is None:
                            row[f'{dcol} {col_suffix} Full']  = float('nan')
                            row[f'{dcol} {col_suffix} Consec'] = float('nan')
                            continue
                        match = dom_subset[dom_subset['nomenclature'] == key]
                        if not match.empty:
                            row[f'{dcol} {col_suffix} Full']  = match['prop_full'].values[0]
                            row[f'{dcol} {col_suffix} Consec'] = match['prop_consec'].values[0]
                        else:
                            row[f'{dcol} {col_suffix} Full']  = float('nan')
                            row[f'{dcol} {col_suffix} Consec'] = float('nan')
                rows.append(row)

        table_df = pd.DataFrame(rows)
        tabular  = build_full_exploration_table(table_df, present_domains)
        latex    = wrap_full_exploration_table(tabular, model_label, variance)

        path = f'figures/tables/full_exploration_table_{model_id}_var{variance}.tex'
        with open(path, 'w') as f:
            f.write(latex)
        print(f'Saved: {path}')

print('\nDone.')

Saved: figures/tables/full_exploration_table_Qwen3-32B_varNo.tex
Saved: figures/tables/full_exploration_table_Qwen3-32B_varLow.tex
Saved: figures/tables/full_exploration_table_Qwen3-32B_varHigh.tex
Saved: figures/tables/full_exploration_table_Olmo-3.1-32B-Instruct_varNo.tex
Saved: figures/tables/full_exploration_table_Olmo-3.1-32B-Instruct_varLow.tex
Saved: figures/tables/full_exploration_table_Olmo-3.1-32B-Instruct_varHigh.tex
Saved: figures/tables/full_exploration_table_gemini-3.1-flash-lite-preview_varNo.tex
Saved: figures/tables/full_exploration_table_gemini-3.1-flash-lite-preview_varLow.tex
Saved: figures/tables/full_exploration_table_gemini-3.1-flash-lite-preview_varHigh.tex

Done.
